# GRPO: от reward к обновлению

Групповые advantages, policy ratio и clipping.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

GRPO сравнивает несколько ответов на один prompt. Reward нормируется внутри группы, чтобы получить advantage. Policy ratio сопоставляет новую политику с политикой, собравшей примеры; clipping ограничивает часть изменений.

В настоящем trainer нужны generation, корректные token masks, сохранённые old log-probs, reference/KL при её использовании и проверенный reward. Ниже только ядро clipped objective без KL и генерации. Это не замена нашему trainer и не запуск TRL. Нулевая дисперсия reward не должна создавать NaN.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
rewards = torch.tensor([[0., 1., 2., 3.], [1., 1., 1., 1.]])
advantages = (rewards - rewards.mean(-1, keepdim=True)) / rewards.std(-1, keepdim=True, unbiased=False).clamp_min(1e-6)
logits = nn.Parameter(torch.zeros(2, 4))
old_logp = logits.detach().log_softmax(-1).clone()
optimizer = torch.optim.Adam([logits], lr=0.08)
for step in range(8):
    logp = logits.log_softmax(-1)
    ratio = (logp - old_logp).exp()
    objective = torch.minimum(ratio * advantages, ratio.clamp(0.8, 1.2) * advantages)
    loss = -objective.mean()
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
print('advantages:', advantages)
print('new probabilities:', logits.softmax(-1).detach())
assert torch.isfinite(loss) and torch.equal(advantages[1], torch.zeros(4))

## Попробуйте сами

Измените reward, включая полностью одинаковые группы. Какие ответы получают большую вероятность? Объясните, почему old_logp нельзя пересчитывать после каждого optimizer.step.

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [DeepSeekMath / GRPO](https://arxiv.org/abs/2402.03300)